# Scalability Analysis

In [ ]:
# Scalability Analysis

from pyspark.ml.classification import RandomForestClassifier as SparkRF

print("Starting scalability experiments")

scalability_results = []

# Strong scaling: same model config, growing data fraction
strong_fracs = [0.05, 0.10, 0.20, 0.40]

print("Strong scaling experiments...")
for frac in strong_fracs:
    subset = train_df.sample(False, frac, seed=42)
    row_count = subset.count()
    rf_s = SparkRF(
        featuresCol="features", labelCol="label",
        numTrees=50, maxDepth=8, maxBins=32, seed=42
    )
    t0 = time.perf_counter()
    rf_s.fit(subset)
    elapsed = round(time.perf_counter() - t0, 2)
    scalability_results.append({
        "experiment": "strong_scaling",
        "data_fraction": frac,
        "row_count": row_count,
        "num_trees": 50,
        "max_depth": 8,
        "train_time_sec": elapsed
    })
    print("Strong | frac:", frac, "| rows:", row_count, "| time:", elapsed, "s")

# Weak scaling: data and trees grow together
print("Weak scaling experiments...")
weak_configs = [
    (0.05,  30,  8),
    (0.10,  60,  8),
    (0.20, 100, 10),
    (0.40, 150, 10),
]

for frac, n_trees, depth in weak_configs:
    subset = train_df.sample(False, frac, seed=42)
    row_count = subset.count()
    rf_w = SparkRF(
        featuresCol="features", labelCol="label",
        numTrees=n_trees, maxDepth=depth, maxBins=32, seed=42
    )
    t0 = time.perf_counter()
    rf_w.fit(subset)
    elapsed = round(time.perf_counter() - t0, 2)
    scalability_results.append({
        "experiment": "weak_scaling",
        "data_fraction": frac,
        "row_count": row_count,
        "num_trees": n_trees,
        "max_depth": depth,
        "train_time_sec": elapsed
    })
    print("Weak | frac:", frac, "| trees:", n_trees, "| rows:", row_count, "| time:", elapsed, "s")

scale_df = pd.DataFrame(scalability_results)
scale_df.to_csv(OUT_DIR + "/dash4_scalability_analysis.csv", index=False)
print("Saved: dash4_scalability_analysis.csv")

# Bottleneck identification table
bottleneck_df = pd.DataFrame([
    {"phase": "Data Ingestion",       "bottleneck_type": "I/O",
     "description": "Parquet read from Google Drive over FUSE mount, single-node I/O bound"},
    {"phase": "Feature Engineering",  "bottleneck_type": "CPU Computation",
     "description": "GeoGrid transformer and 8 interaction columns, CPU bound on driver"},
    {"phase": "RF Model Training",    "bottleneck_type": "CPU Computation",
     "description": "Tree splitting across 200 shuffle partitions, no GPU utilisation"},
    {"phase": "CrossValidator / TVS", "bottleneck_type": "CPU Computation",
     "description": "Sequential fits on subsample, parallelism=1 to avoid OOM on Colab"},
    {"phase": "toPandas collect",     "bottleneck_type": "Network / Memory",
     "description": "Collecting test predictions to driver for sklearn metrics"},
    {"phase": "CSV export to Drive",  "bottleneck_type": "I/O",
     "description": "coalesce(1) write creates single file, slow for large partitions"},
])
bottleneck_df.to_csv(OUT_DIR + "/dash4_bottleneck_notes.csv", index=False)
print("Saved: dash4_bottleneck_notes.csv")

Starting scalability experiments
Strong scaling experiments...
Strong | frac: 0.05 | rows: 48041 | time: 18.4 s
Strong | frac: 0.1 | rows: 95866 | time: 28.2 s
Strong | frac: 0.2 | rows: 192753 | time: 45.59 s
Strong | frac: 0.4 | rows: 385624 | time: 78.81 s
Weak scaling experiments...
Weak | frac: 0.05 | trees: 30 | rows: 48041 | time: 10.71 s
Weak | frac: 0.1 | trees: 60 | rows: 95866 | time: 36.89 s
Weak | frac: 0.2 | trees: 100 | rows: 192753 | time: 194.26 s
Weak | frac: 0.4 | trees: 150 | rows: 385624 | time: 550.42 s
Saved: dash4_scalability_analysis.csv
Saved: dash4_bottleneck_notes.csv


#  Statistical Significance and Stability Analysis


In [ ]:
#  Statistical Significance and Stability Analysis

best_pred_pd = rf_pred.select(
    F.col("label").cast("int").alias("label"),
    F.col("prediction").cast("int").alias("prediction")
).toPandas()

best_pred_pd["correct"] = (
    best_pred_pd["label"] == best_pred_pd["prediction"]
).astype(int)

n_bootstrap = 500
rng         = np.random.default_rng(42)
boot_accs   = []

for i in range(n_bootstrap):
    idx    = rng.integers(0, len(best_pred_pd), size=len(best_pred_pd))
    sample = best_pred_pd["correct"].iloc[idx]
    boot_accs.append(float(sample.mean()))

boot_accs = np.array(boot_accs)
ci_lower  = round(float(np.percentile(boot_accs, 2.5)),  4)
ci_upper  = round(float(np.percentile(boot_accs, 97.5)), 4)
ci_mean   = round(float(np.mean(boot_accs)), 4)

print("Bootstrap 500 iterations - Random Forest (best model):")
print("  Mean accuracy :", ci_mean)
print("  95 pct CI lower:", ci_lower)
print("  95 pct CI upper:", ci_upper)

bootstrap_df = pd.DataFrame([{
    "model":        "Random Forest (best)",
    "n_bootstrap":  n_bootstrap,
    "mean_accuracy": ci_mean,
    "ci_lower_95":  ci_lower,
    "ci_upper_95":  ci_upper,
    "ci_width":     round(ci_upper - ci_lower, 4)
}])
bootstrap_df.to_csv(OUT_DIR + "/dash2_bootstrap_ci.csv", index=False)
print("Saved: dash2_bootstrap_ci.csv")


# measure variance in accuracy and F1 across trials
print("Running stability analysis across 5 trials...")

stability_rows = []

for trial in range(5):
    seed_val  = 200 + trial * 13
    sub       = train_df.sample(False, 0.15, seed=seed_val)
    rf_stab   = SparkRF(
        featuresCol="features", labelCol="label",
        numTrees=80, maxDepth=10, maxBins=32, seed=seed_val
    )
    stab_model = rf_stab.fit(sub)
    stab_pred  = stab_model.transform(test_df)
    stab_acc   = round(float(acc_eval.evaluate(stab_pred)), 4)
    stab_f1    = round(float(f1_eval.evaluate(stab_pred)),  4)
    stability_rows.append({
        "trial":    trial + 1,
        "seed":     seed_val,
        "train_fraction": 0.15,
        "accuracy": stab_acc,
        "f1_score": stab_f1
    })
    print("Trial", trial + 1, "| seed:", seed_val,
          "| acc:", stab_acc, "| f1:", stab_f1)

stability_df = pd.DataFrame(stability_rows)
acc_std = round(float(stability_df["accuracy"].std()), 4)
f1_std  = round(float(stability_df["f1_score"].std()),  4)

stability_df["acc_std_across_trials"] = acc_std
stability_df["f1_std_across_trials"]  = f1_std

stability_df.to_csv(OUT_DIR + "/dash2_stability_analysis.csv", index=False)
print("Saved: dash2_stability_analysis.csv")
print("Accuracy std across 5 trials:", acc_std)
print("F1 std across 5 trials:      ", f1_std)
print("Low std = stable model, high std = sensitive to data perturbations")

Bootstrap 500 iterations - Random Forest (best model):
  Mean accuracy : 0.6028
  95 pct CI lower: 0.6011
  95 pct CI upper: 0.6047
Saved: dash2_bootstrap_ci.csv
Running stability analysis across 5 trials...
Trial 1 | seed: 200 | acc: 0.6028 | f1: 0.4761
Trial 2 | seed: 213 | acc: 0.6036 | f1: 0.4812
Trial 3 | seed: 226 | acc: 0.6028 | f1: 0.4767
Trial 4 | seed: 239 | acc: 0.6028 | f1: 0.4746
Trial 5 | seed: 252 | acc: 0.6026 | f1: 0.4769
Saved: dash2_stability_analysis.csv
Accuracy std across 5 trials: 0.0004
F1 std across 5 trials:       0.0025
Low std = stable model, high std = sensitive to data perturbations


In [ ]:
# Statistical Significance and Stability Analysis
# Bootstrap CI: required by brief for statistical significance testing

print("Running bootstrap confidence intervals...")

best_pred_pd = rf_pred.select(
    F.col("label").cast("int").alias("label"),
    F.col("prediction").cast("int").alias("prediction")
).toPandas()

best_pred_pd["correct"] = (
    best_pred_pd["label"] == best_pred_pd["prediction"]
).astype(int)

n_bootstrap = 500
rng         = np.random.default_rng(42)
boot_accs   = []

for i in range(n_bootstrap):
    idx    = rng.integers(0, len(best_pred_pd), size=len(best_pred_pd))
    sample = best_pred_pd["correct"].iloc[idx]
    boot_accs.append(float(sample.mean()))

boot_accs = np.array(boot_accs)
ci_lower  = round(float(np.percentile(boot_accs, 2.5)),  4)
ci_upper  = round(float(np.percentile(boot_accs, 97.5)), 4)
ci_mean   = round(float(np.mean(boot_accs)), 4)

print("Bootstrap 500 iterations - Random Forest (best model):")
print("  Mean accuracy :", ci_mean)
print("  95 pct CI lower:", ci_lower)
print("  95 pct CI upper:", ci_upper)

bootstrap_df = pd.DataFrame([{
    "model":        "Random Forest (best)",
    "n_bootstrap":  n_bootstrap,
    "mean_accuracy": ci_mean,
    "ci_lower_95":  ci_lower,
    "ci_upper_95":  ci_upper,
    "ci_width":     round(ci_upper - ci_lower, 4)
}])
bootstrap_df.to_csv(OUT_DIR + "/dash2_bootstrap_ci.csv", index=False)
print("Saved: dash2_bootstrap_ci.csv")

# Stability Analysis: train 5 models on slightly different subsamples
# measure variance in accuracy and F1 across trials
print("Running stability analysis across 5 trials.")

stability_rows = []

for trial in range(5):
    seed_val  = 200 + trial * 13
    sub       = train_df.sample(False, 0.15, seed=seed_val)
    rf_stab   = SparkRF(
        featuresCol="features", labelCol="label",
        numTrees=80, maxDepth=10, maxBins=32, seed=seed_val
    )
    stab_model = rf_stab.fit(sub)
    stab_pred  = stab_model.transform(test_df)
    stab_acc   = round(float(acc_eval.evaluate(stab_pred)), 4)
    stab_f1    = round(float(f1_eval.evaluate(stab_pred)),  4)
    stability_rows.append({
        "trial":    trial + 1,
        "seed":     seed_val,
        "train_fraction": 0.15,
        "accuracy": stab_acc,
        "f1_score": stab_f1
    })
    print("Trial", trial + 1, "| seed:", seed_val,
          "| acc:", stab_acc, "| f1:", stab_f1)

stability_df = pd.DataFrame(stability_rows)
acc_std = round(float(stability_df["accuracy"].std()), 4)
f1_std  = round(float(stability_df["f1_score"].std()),  4)

stability_df["acc_std_across_trials"] = acc_std
stability_df["f1_std_across_trials"]  = f1_std

stability_df.to_csv(OUT_DIR + "/dash2_stability_analysis.csv", index=False)
print("Saved: dash2_stability_analysis.csv")
print("Accuracy std across 5 trials:", acc_std)
print("F1 std across 5 trials:      ", f1_std)
print("Low std = stable model, high std = sensitive to data perturbations")

Running bootstrap confidence intervals...
Bootstrap 500 iterations - Random Forest (best model):
  Mean accuracy : 0.6028
  95 pct CI lower: 0.6011
  95 pct CI upper: 0.6047
Saved: dash2_bootstrap_ci.csv
Running stability analysis across 5 trials.
Trial 1 | seed: 200 | acc: 0.6028 | f1: 0.4761
Trial 2 | seed: 213 | acc: 0.6036 | f1: 0.4812
Trial 3 | seed: 226 | acc: 0.6028 | f1: 0.4767
Trial 4 | seed: 239 | acc: 0.6028 | f1: 0.4746
Trial 5 | seed: 252 | acc: 0.6026 | f1: 0.4769
Saved: dash2_stability_analysis.csv
Accuracy std across 5 trials: 0.0004
F1 std across 5 trials:       0.0025
Low std = stable model, high std = sensitive to data perturbations


In [ ]:
# Statistical Significance and Stability Analysis

print("Running bootstrap confidence intervals...")

best_pred_pd = rf_pred.select(
    F.col("label").cast("int").alias("label"),
    F.col("prediction").cast("int").alias("prediction")
).toPandas()

best_pred_pd["correct"] = (
    best_pred_pd["label"] == best_pred_pd["prediction"]
).astype(int)

n_bootstrap = 500
rng         = np.random.default_rng(42)
boot_accs   = []

for i in range(n_bootstrap):
    idx    = rng.integers(0, len(best_pred_pd), size=len(best_pred_pd))
    sample = best_pred_pd["correct"].iloc[idx]
    boot_accs.append(float(sample.mean()))

boot_accs = np.array(boot_accs)
ci_lower  = round(float(np.percentile(boot_accs, 2.5)),  4)
ci_upper  = round(float(np.percentile(boot_accs, 97.5)), 4)
ci_mean   = round(float(np.mean(boot_accs)), 4)

print("Bootstrap 500 iterations - Random Forest (best model):")
print("  Mean accuracy :", ci_mean)
print("  95 pct CI lower:", ci_lower)
print("  95 pct CI upper:", ci_upper)

bootstrap_df = pd.DataFrame([{
    "model":        "Random Forest (best)",
    "n_bootstrap":  n_bootstrap,
    "mean_accuracy": ci_mean,
    "ci_lower_95":  ci_lower,
    "ci_upper_95":  ci_upper,
    "ci_width":     round(ci_upper - ci_lower, 4)
}])
bootstrap_df.to_csv(OUT_DIR + "/dash2_bootstrap_ci.csv", index=False)
print("Saved: dash2_bootstrap_ci.csv")

# Stability Analysis: train 5 models on slightly different subsamples
# measure variance in accuracy and F1 across trials

stability_rows = []

for trial in range(5):
    seed_val  = 200 + trial * 13
    sub       = train_df.sample(False, 0.15, seed=seed_val)
    rf_stab   = SparkRF(
        featuresCol="features", labelCol="label",
        numTrees=80, maxDepth=10, maxBins=32, seed=seed_val
    )
    stab_model = rf_stab.fit(sub)
    stab_pred  = stab_model.transform(test_df)
    stab_acc   = round(float(acc_eval.evaluate(stab_pred)), 4)
    stab_f1    = round(float(f1_eval.evaluate(stab_pred)),  4)
    stability_rows.append({
        "trial":    trial + 1,
        "seed":     seed_val,
        "train_fraction": 0.15,
        "accuracy": stab_acc,
        "f1_score": stab_f1
    })
    print("Trial", trial + 1, "| seed:", seed_val,
          "| acc:", stab_acc, "| f1:", stab_f1)

stability_df = pd.DataFrame(stability_rows)
acc_std = round(float(stability_df["accuracy"].std()), 4)
f1_std  = round(float(stability_df["f1_score"].std()),  4)

stability_df["acc_std_across_trials"] = acc_std
stability_df["f1_std_across_trials"]  = f1_std

stability_df.to_csv(OUT_DIR + "/dash2_stability_analysis.csv", index=False)
print("Saved: dash2_stability_analysis.csv")
print("Accuracy std across 5 trials:", acc_std)
print("F1 std across 5 trials:      ", f1_std)
print("Low std = stable model, high std = sensitive to data perturbations")

Running bootstrap confidence intervals...
Bootstrap 500 iterations - Random Forest (best model):
  Mean accuracy : 0.6028
  95 pct CI lower: 0.6011
  95 pct CI upper: 0.6047
Saved: dash2_bootstrap_ci.csv
Trial 1 | seed: 200 | acc: 0.6028 | f1: 0.4761
Trial 2 | seed: 213 | acc: 0.6036 | f1: 0.4812
Trial 3 | seed: 226 | acc: 0.6028 | f1: 0.4767
Trial 4 | seed: 239 | acc: 0.6028 | f1: 0.4746
Trial 5 | seed: 252 | acc: 0.6026 | f1: 0.4769
Saved: dash2_stability_analysis.csv
Accuracy std across 5 trials: 0.0004
F1 std across 5 trials:       0.0025
Low std = stable model, high std = sensitive to data perturbations


In [ ]:
# Dashboard 3 (Business Insights) and Dashboard 4


# Accuracy by district
(rf_pred
    .withColumn("correct", (F.col("label") == F.col("prediction")).cast("int"))
    .groupBy("District")
    .agg(
        F.count("*").alias("total_predictions"),
        F.sum("correct").alias("correct_predictions"),
        F.round(F.avg("correct"), 4).alias("district_accuracy")
    )
    .orderBy(F.desc("district_accuracy"))
    .coalesce(1).write.mode("overwrite").option("header", True)
    .csv(OUT_DIR + "/dash3_accuracy_by_district")
)
print("Saved: dash3_accuracy_by_district")

# Predicted crime category by hour - for patrol deployment planning
(rf_pred
    .groupBy("hour", "prediction")
    .count()
    .withColumnRenamed("count", "crime_count")
    .orderBy("hour", F.desc("crime_count"))
    .coalesce(1).write.mode("overwrite").option("header", True)
    .csv(OUT_DIR + "/dash3_predictions_by_hour")
)
print("Saved: dash3_predictions_by_hour")

# Weekend vs weekday breakdown
(rf_pred
    .groupBy("is_weekend", "prediction")
    .count()
    .orderBy("is_weekend", F.desc("count"))
    .coalesce(1).write.mode("overwrite").option("header", True)
    .csv(OUT_DIR + "/dash3_weekend_vs_weekday")
)
print("Saved: dash3_weekend_vs_weekday")

# Geographic prediction density for map viz in Tableau
(rf_pred
    .withColumn("lat_r", F.round("Latitude",  2))
    .withColumn("lon_r", F.round("Longitude", 2))
    .groupBy("lat_r", "lon_r", "prediction")
    .count()
    .orderBy(F.desc("count"))
    .limit(5000)
    .coalesce(1).write.mode("overwrite").option("header", True)
    .csv(OUT_DIR + "/dash3_geo_prediction_density")
)
print("Saved: dash3_geo_prediction_density")

# Seasonal crime prediction breakdown
(rf_pred
    .groupBy("season", "prediction")
    .count()
    .orderBy("season", F.desc("count"))
    .coalesce(1).write.mode("overwrite").option("header", True)
    .csv(OUT_DIR + "/dash3_predictions_by_season")
)
print("Saved: dash3_predictions_by_season")

print("Exporting Dashboard 4 data...")

# Cost-performance tradeoff table
cost_perf_df = pd.DataFrame([
    {"model": "Logistic Regression", "framework": "PySpark MLlib",
     "accuracy": round(lr_acc, 4),   "f1_score": round(lr_f1, 4),
     "train_time_sec": round(lr_time, 2),
     "cost_index": round(lr_time / lr_acc, 2)},
    {"model": "Decision Tree",       "framework": "PySpark MLlib",
     "accuracy": round(dt_acc, 4),   "f1_score": round(dt_f1, 4),
     "train_time_sec": round(dt_time, 2),
     "cost_index": round(dt_time / dt_acc, 2)},
    {"model": "Random Forest",       "framework": "PySpark MLlib",
     "accuracy": round(rf_acc, 4),   "f1_score": round(rf_f1, 4),
     "train_time_sec": round(rf_time, 2),
     "cost_index": round(rf_time / rf_acc, 2)},
    {"model": "GBT (Binary)",        "framework": "PySpark MLlib",
     "accuracy": round(gbt_acc, 4),  "f1_score": round(gbt_f1, 4),
     "train_time_sec": round(gbt_time, 2),
     "cost_index": round(gbt_time / gbt_acc, 2)},
    {"model": best_model_name + " Tuned", "framework": "PySpark MLlib",
     "accuracy": round(tuned_acc, 4), "f1_score": round(tuned_f1, 4),
     "train_time_sec": round(cv_time, 2),
     "cost_index": round(cv_time / tuned_acc, 2)},
    {"model": "Sklearn LR",          "framework": "Sklearn single-node",
     "accuracy": sk_lr_acc,          "f1_score": sk_lr_f1,
     "train_time_sec": round(sk_lr_time, 2),
     "cost_index": round(sk_lr_time / max(sk_lr_acc, 0.0001), 2)},
    {"model": "Sklearn DT",          "framework": "Sklearn single-node",
     "accuracy": sk_dt_acc,          "f1_score": sk_dt_f1,
     "train_time_sec": round(sk_dt_time, 2),
     "cost_index": round(sk_dt_time / max(sk_dt_acc, 0.0001), 2)},
    {"model": "Sklearn RF",          "framework": "Sklearn single-node",
     "accuracy": sk_rf_acc,          "f1_score": sk_rf_f1,
     "train_time_sec": round(sk_rf_time, 2),
     "cost_index": round(sk_rf_time / max(sk_rf_acc, 0.0001), 2)},
])
cost_perf_df.to_csv(OUT_DIR + "/dash4_cost_performance.csv", index=False)
print("Saved: dash4_cost_performance.csv")
print(cost_perf_df.to_string(index=False))

Saved: dash3_accuracy_by_district
Saved: dash3_predictions_by_hour
Saved: dash3_weekend_vs_weekday
Saved: dash3_geo_prediction_density
Saved: dash3_predictions_by_season
Exporting Dashboard 4 data...
Saved: dash4_cost_performance.csv
              model           framework  accuracy  f1_score  train_time_sec  cost_index
Logistic Regression       PySpark MLlib    0.5992    0.4527           75.58      126.13
      Decision Tree       PySpark MLlib    0.5984    0.5316           52.20       87.23
      Random Forest       PySpark MLlib    0.6029    0.4869         4004.04     6641.75
       GBT (Binary)       PySpark MLlib    0.6931    0.5902          258.78      373.36
 RandomForest Tuned       PySpark MLlib    0.6029    0.4869          126.46      209.78
         Sklearn LR Sklearn single-node    0.5994    0.4563            6.93       11.56
         Sklearn DT Sklearn single-node    0.5613    0.5301            0.98        1.74
         Sklearn RF Sklearn single-node    0.6020    0.5300   